# 07. Rotary Positional Encoding (RoPE): Math & Implementation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/The-Pocket/PocketFlow-Tutorial-Video-Generator/blob/main/docs/llm/07_Rotary_Positional_Encoding_RoPE.ipynb)

> **Tutorial Overview**: Master Rotary Positional Embeddings (RoPE) used in LLaMA, Mistral, and DeepSeek. Learn how 2D complex rotations inject relative positional awareness into self-attention.
> **Original Source**: `rope.md`

---


In [ ]:
# Setup & Imports
!pip install -q torch matplotlib numpy

import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)


# **Title: Rotary Positional Encoding (RoPE): A Deep Dive**

## **Introduction: Give me 40 minutes, and I will make RoPE click forever.**

You have likely heard that modern Large Language Models like Llama, PaLM, and GPT-NeoX have abandoned traditional positional embeddings. Instead, they use a more powerful and elegant method: **Rotary Positional Encoding**, or **RoPE**.

What if I told you the entire "magic" behind this technique is just high-school trigonometry? What if the core of this powerful idea is captured in this single matrix operation?

$$
\begin{pmatrix} x'_0 \\ x'_1 \end{pmatrix} = \begin{pmatrix} \cos(m\theta_0) & -\sin(m\theta_0) \\ \sin(m\theta_0) & \cos(m\theta_0) \end{pmatrix} \begin{pmatrix} x_0 \\ x_1 \end{pmatrix}
$$

This is the 2D rotation matrix. It's the entire secret. By the end of this tutorial, this formula will not only make sense, but you will understand how it's generalized to high dimensions and why it's the key to unlocking **relative positional information** in Transformers.

#### **Our Promise**

Our promise is simple: in the next 40 minutes, you will understand RoPE from first principles to practical implementation. You will understand not only **what** the algorithm is, but **why** it works so effectively. By the end, you will understand this code:


In [ ]:
def apply_rotary_pos_emb(x: torch.Tensor, rope_emb: torch.Tensor):
    seq_len = x.shape[1]
    rope_emb_sliced = rope_emb[:seq_len, :].unsqueeze(0).unsqueeze(2)
    cos_emb = rope_emb_sliced.cos()
    sin_emb = rope_emb_sliced.sin()

    x_reshaped = x.float().reshape(*x.shape[:-1], -1, 2)
    x_partner = torch.stack([-x_reshaped[..., 1], x_reshaped[..., 0]], dim=-1)
    x_partner = x_partner.flatten(-2)

    return (x * cos_emb + x_partner * sin_emb).type_as(x)


This is the complete RoPE implementation. Every line will make sense by the end.

## **Chapter 1: The Problem: The Limitations of Absolute Positions**

To understand why RoPE is a breakthrough, we must first understand the method it replaced: **Absolute Positional Embeddings**.

#### **The Old Way: Assigning an "Address" to Each Position**

In early Transformer models like BERT and GPT-2, the position of a token was handled by creating a unique vector for each possible position, up to a maximum length (e.g., 1024).

1.  A token's meaning is represented by its **Token Embedding**.
2.  A token's location is represented by its **Positional Embedding**.
3.  The final input vector is the sum: `Input Vector = Token Embedding + Positional Embedding`.

This is like giving each word in a sentence a specific street address.

| Word | Token Embedding | Position | Positional Embedding | Final Input Vector |
| :--- | :--- | :--- | :--- | :--- |
| "The" | `vec("The")` | 0 | `vec(pos=0)` | `vec("The") + vec(pos=0)` |
| "red" | `vec("red")` | 1 | `vec(pos=1)` | `vec("red") + vec(pos=1)` |
| "car" | `vec("car")` | 2 | `vec(pos=2)` | `vec("car") + vec(pos=2)` |

#### **The Flaw: Context is Relative, but Addresses are Absolute**

Language is built on relative relationships. The meaning of "red car" doesn't change based on where it appears in a document. However, the absolute embedding method fundamentally changes the input vectors.

Consider these two sentences:
1.  "**The red car** is fast."
2.  "I saw **the red car**."

Let's look at the final vector for the word "red" in each sentence.

*   In sentence 1, "red" is at position 1. Its final vector is `vec("red") + vec(pos=1)`.
*   In sentence 2, "red" is at position 3. Its final vector is `vec("red") + vec(pos=3)`.

These are two different vectors. The model receives a different input for the exact same word, simply because its absolute position changed. The attention mechanism now has a harder job. It must learn from scratch that the relationship between `vec("red") + vec(pos=1)` and `vec("car") + vec(pos=2)` is the same as the relationship between `vec("red") + vec(pos=3)` and `vec("car") + vec(pos=4)`.

The model doesn't inherently know that "position 4" is one step away from "position 3". It only knows that `vec(pos=3)` and `vec(pos=4)` are two distinct, arbitrary vectors that it needs to learn the relationship between. This is computationally expensive and doesn't generalize well to positions the model hasn't seen during training.

#### **The Goal: A New System Based on Relative Distance**

We need an encoding scheme that bakes the concept of relative position directly into the math. Ideally, the attention score between a query vector `q` at position `m` and a key vector `k` at position `n` should be computable from a function that looks like this:

`Score = f(q, k, m-n)`

The score should depend on the vectors themselves and their **relative distance `m-n`**, not their absolute positions `m` and `n`.

This is the problem RoPE solves. It provides a way to modify `q` and `k` such that their dot product naturally produces this desired relative relationship. The solution, as we will see, is found not in adding vectors, but in rotating them.

Before we dive in, let's clarify exactly how RoPE fits into the Transformer:

| Question | Answer |
|:---|:---|
| **Applied to which vectors?** | Only Q and K, not V. We need position in the attention score (`QK^T`), not in the output values. |
| **Is RoPE itself learned?** | No. The rotation angles are fixed formulas based on position. But the Q and K projection weights (Wq, Wk) *are* learned, and they learn to produce vectors that work well with these rotations. |
| **Applied in every layer?** | Yes. Every transformer block applies RoPE to its Q and K vectors independently. |

## **Chapter 2: The Core Intuition: Encoding Position via Rotation (in 2D)**

We need a transformation that modifies a vector to encode its position while preserving its original information (its meaning). The key insight of RoPE is that a **rotation** does exactly this.

A rotation changes a vector's direction but, crucially, **it does not change its length (norm)**. We can use the original length to represent the token's meaning and the new direction to represent its position.

#### **The Building Block: 2D Rotation**

Let's start in two dimensions. Imagine a token's meaning is captured by a simple 2D vector, `v = (x, y)`. To rotate this vector by an angle `θ`, we multiply it by the standard 2D rotation matrix:

$$
R(\theta) = \begin{pmatrix} \cos(\theta) & -\sin(\theta) \\ \sin(\theta) & \cos(\theta) \end{pmatrix}
$$

The new, rotated vector `v'` is calculated as:

$$
v' = R(\theta)v = \begin{pmatrix} \cos(\theta) & -\sin(\theta) \\ \sin(\theta) & \cos(\theta) \end{pmatrix} \begin{pmatrix} x \\ y \end{pmatrix}
$$

#### **A Concrete Example: Rotation by 90 Degrees**

Let's make this tangible. Suppose our vector is `v = (1, 2)` and we want to rotate it by `θ = 90°` (or `π/2` radians).

1.  **The Angle:** We know `cos(90°) = 0` and `sin(90°) = 1`.

2.  **The Rotation Matrix:** Plugging these values into `R(θ)` gives us:
    $$
    R(90^{\circ}) = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}
    $$

3.  **The Calculation:** Now we perform the matrix multiplication:
    $$
    v' = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix} \begin{pmatrix} 1 \\ 2 \end{pmatrix} = \begin{pmatrix} (0 \cdot 1) + (-1 \cdot 2) \\ (1 \cdot 1) + (0 \cdot 2) \end{pmatrix} = \begin{pmatrix} -2 \\ 1 \end{pmatrix}
    $$
    Our new vector is `v' = (-2, 1)`.

4.  **Verifying the Length:**
    *   Length of original vector `v`: `||v|| = sqrt(1² + 2²) = sqrt(5)`
    *   Length of rotated vector `v'`: `||v'|| = sqrt((-2)² + 1²) = sqrt(4 + 1) = sqrt(5)`
    The length is perfectly preserved. All we changed was the direction.


```text
A 2D coordinate plane.
The x-axis goes from -3 to 3. The y-axis goes from -3 to 3.
Vector v starts at the origin (0,0) and points to the dot (1, 2). It's an arrow in the first quadrant.
Vector v' starts at the origin (0,0) and points to the dot (-2, 1). It's an arrow in the second quadrant.
An arc with an arrow shows the counter-clockwise 90-degree rotation from v to v'.
```


#### **The "Aha!" Moment: Connecting Rotation to Position**

Here is the core idea of RoPE: **The angle of rotation is determined by the token's position `m`**.

We define a constant, base "frequency" or angle, `θ`. The total rotation applied to a vector at position `m` is simply `m * θ`.

| Position (`m`) | Total Rotation Angle | What it means |
| :--- | :--- | :--- |
| 0 | `0 * θ = 0` | The vector for the first token is **not rotated**. It is our baseline. |
| 1 | `1 * θ = θ` | The vector for the second token is rotated by a small angle `θ`. |
| 2 | `2 * θ` | The vector for the third token is rotated by twice that angle. |
| 3 | `3 * θ` | The vector for the fourth token is rotated by three times that angle. |

This establishes a clear, consistent rule: the further a token is in the sequence, the more its vector is "spun" around the origin.


```text
A 3D visualization.
The X and Y axes form a 2D plane at the bottom. The Z axis represents the position 'm' and goes upwards.
At z=0 (position 0), a vector 'v' points from the origin along the positive X-axis.
At z=1 (position 1), the same vector is shown, but rotated slightly counter-clockwise in the XY plane.
At z=2 (position 2), the vector is rotated even more.
At z=3 (position 3), it's rotated further still.
A dotted line connects the tips of these vectors, forming a spiral or helix shape that winds upwards along the Z-axis.
This image shows how the vector's direction in the XY plane progressively changes as its position 'm' increases.
```
